In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
movies_df = (
    spark.read
    .table("movie_silver.movies")
    .filter(f"file_date = '{v_file_date}'")
    
)

In [0]:
languages_df = (
    spark.read
        .table("movie_silver.languages")
)

In [0]:
movie_languages_df = (
    spark.read
        .table("movie_silver.movies_languages")
        .filter(f"file_date = '{v_file_date}'")
)

In [0]:
genres_df = (
    spark.read
        .table("movie_silver.genres")
)

In [0]:
movies_genres_df = (
    spark.read
        .table("movie_silver.movie_genre")
        .filter(f"file_date = '{v_file_date}'")
)

In [0]:
from pyspark.sql.functions import col, desc, asc, lit

In [0]:
results_movie_genre_language_df = (
    movies_df.filter(col("year_release_date") >= 2000)
        .withColumn("created_date", lit(v_file_date))
        .join(movie_languages_df, 
              movies_df["movie_id"] == movie_languages_df["movie_id"], "inner")
        .join(languages_df,
              movie_languages_df["language_id"] == languages_df["language_id"], "inner")
        .join(movies_genres_df, 
              movies_df["movie_id"] == movies_genres_df["movie_id"], "inner")
        .join(genres_df,
              movies_genres_df["genre_id"] == genres_df["genre_id"], "inner")
).select(
    movies_df["title"], movies_df["duration_time"], movies_df["release_date"], movies_df["vote_average"],
    languages_df["language_name"],
    genres_df["genre_name"],
    col("created_date")
).orderBy(
    col("release_date").asc()
)

In [0]:
display(results_movie_genre_language_df)

In [0]:
overwrite_partitioned_table("movie_gold","results_movie_genre_language","created_date", v_file_date)

In [0]:
(
    results_movie_genre_language_df
    .write
        .mode("append")
        .partitionBy("created_date")
        .saveAsTable(f"movie_gold.results_movie_genre_language")
)

In [0]:
%sql
select * from movie_gold.results_movie_genre_language

In [0]:
%sql
SELECT created_date,
COUNT(*)
FROM movie_gold.results_movie_genre_language
GROUP BY created_date